
# RDD2022 — YOLOv11n-CDL Low-Light Robustness
## Reproduce / Validate Saved Results (No Retraining)

This notebook is the **reproduction/validation notebook** for the project.

### Purpose

It reproduces the reported R1–R5 evaluation results using the artifacts already saved in this repository:

- frozen RDD2022 split
- `test_dark_mild`
- `test_dark_severe`
- saved `best.pt` checkpoints
- saved CSV result tables

**This notebook does not train any model and does not regenerate the dark test sets.**

The second notebook in the repository,

`RDD2022_YOLOv11n_CDL+Adaptive_AGE_full_experiment_from_scratch.ipynb`

is the full experiment notebook and contains the training/data-preparation pipeline.

### Model ladder

| Variant | Description |
|---|---|
| R1 | Stock YOLOv11n baseline |
| R2 | YOLOv11n + ConvSmart + DSA + P2 |
| R3 | R2 + low-light training augmentation |
| R4 | R3 + Adaptive Gamma Enhancement (AGE) |
| R5 | R3 + brightness-gated AGE |

> **Important:** a fresh training run is not expected to reproduce the exact reported values because training is stochastic. This notebook instead validates the **saved experiment artifacts**.


## 1. Repository setup

In [ ]:

# This notebook is intended to run from the GitHub repository root.
# If it is opened elsewhere, set REPO_ROOT manually before continuing.

from pathlib import Path
import os, sys, json, re, glob, shutil, subprocess, importlib

def find_repo_root():
    candidates = [
        Path.cwd(),
        Path("/content/YOLOv11n-CDL-RDD2022-LowLight"),
        Path("/content"),
    ]
    for p in candidates:
        if (p / "common.py").exists() and (p / "runs").exists():
            return p.resolve()
    return Path.cwd().resolve()

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)

print("Repository root:", REPO_ROOT)
print("Repository contents:")
for p in sorted(REPO_ROOT.iterdir(), key=lambda x: x.name.lower()):
    print("  ", p.name)


In [ ]:

# Install the versions used by the experiment.
# This cell installs dependencies only; it does NOT train or evaluate anything.

%pip install -q "ultralytics==8.3.152" "albumentations==1.4.18" --no-deps
%pip install -q "albucore==0.0.17" "eval-type-backport"


In [ ]:

import torch
import torchvision
import ultralytics
import albumentations

print("ultralytics:", ultralytics.__version__)
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("albumentations:", albumentations.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Load the saved project artifacts

In [ ]:

# Expected repository artifacts.
FROZEN_DIR = REPO_ROOT / "frozen_split"
DARK_MILD_DIR = REPO_ROOT / "test_dark_mild"
DARK_SEVERE_DIR = REPO_ROOT / "test_dark_severe"
RUNS_DIR = REPO_ROOT / "runs"

required_dirs = [FROZEN_DIR, DARK_MILD_DIR, DARK_SEVERE_DIR, RUNS_DIR]
for d in required_dirs:
    print(f"{d.name:20s}:", "OK" if d.exists() else "MISSING")

# The frozen dataset is stored as an LFS-tracked ZIP in the repository.
FROZEN_ZIP = FROZEN_DIR / "rdd2022_yolo_4k.zip"
print("Frozen ZIP:", FROZEN_ZIP, "->", "OK" if FROZEN_ZIP.exists() else "MISSING")


In [ ]:

# Find the frozen test images without changing the saved split.
# The ZIP is extracted only to a temporary working directory if necessary.
import zipfile

WORK_DIR = REPO_ROOT / "_reproduction_work"
YOLO_DIR = WORK_DIR / "yolo"

if not (YOLO_DIR / "images" / "test").exists():
    if not FROZEN_ZIP.exists():
        raise FileNotFoundError(
            f"Missing {FROZEN_ZIP}. The repository must contain the frozen split ZIP."
        )
    WORK_DIR.mkdir(exist_ok=True)
    print("Extracting frozen split to:", YOLO_DIR)
    with zipfile.ZipFile(FROZEN_ZIP, "r") as z:
        z.extractall(WORK_DIR)

# Locate the actual YOLO root in case the ZIP contains an extra directory.
possible = [
    YOLO_DIR,
    WORK_DIR / "rdd2022_yolo_4k",
]
if not (YOLO_DIR / "images" / "test").exists():
    for p in WORK_DIR.rglob("images/test"):
        YOLO_DIR = p.parent.parent
        break

print("YOLO data root:", YOLO_DIR)
print("Test images:", len(list((YOLO_DIR / "images" / "test").glob("*.jpg"))))


In [ ]:

import yaml

DATA_DIR = WORK_DIR / "yaml"
DATA_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_YAML = DATA_DIR / "rdd2022_clean.yaml"
CLEAN_YAML.write_text(f'''path: {YOLO_DIR.as_posix()}
train: images/train
val: images/val
test: images/test
names:
  0: D00
  1: D10
  2: D20
  3: D40
''')

def write_dark_yaml(name, directory):
    path = DATA_DIR / f"{name}.yaml"
    path.write_text(f'''path: {directory.as_posix()}
train: images
val: images
test: images
names:
  0: D00
  1: D10
  2: D20
  3: D40
''')
    return path

MILD_YAML = write_dark_yaml("dark_mild", DARK_MILD_DIR)
SEVERE_YAML = write_dark_yaml("dark_severe", DARK_SEVERE_DIR)

print(CLEAN_YAML)
print(MILD_YAML)
print(SEVERE_YAML)


## 3. Register the project's custom modules

In [ ]:

# The saved checkpoints for R2-R5 use project-specific modules.
# common.py is the tracked source of truth for ConvSmart, DSA and utilities.

sys.path.insert(0, str(REPO_ROOT))
import common
importlib.reload(common)

print("Loaded:", REPO_ROOT / "common.py")
print("Available registration helper:", hasattr(common, "register_custom_modules"))

if hasattr(common, "register_custom_modules"):
    common.register_custom_modules()
    print("Custom modules registered.")



### AGE compatibility

R4 and R5 both use a module named `AGE`, but they use different forward behaviour:

- **R4:** naive AGE, `gamma = 0.5 + sigmoid(...)`
- **R5:** brightness-gated AGE, where the correction is blended toward identity according to image brightness.

The reproduction notebook registers the appropriate implementation **before loading each checkpoint** so that the saved R4 and R5 models are evaluated with the same AGE definitions used by the experiment.


In [ ]:

import torch.nn as nn
import torch.nn.functional as F
import ultralytics.nn.tasks as tasks

class NaiveAGE(nn.Module):
    def __init__(self):
        super().__init__()
        self.gamma_net = nn.Sequential(
            nn.Conv2d(3, 16, 3, 2, 1), nn.LeakyReLU(inplace=True),
            nn.Conv2d(16, 32, 3, 2, 1), nn.LeakyReLU(inplace=True),
            nn.Conv2d(32, 32, 3, 2, 1), nn.LeakyReLU(inplace=True),
            nn.Conv2d(32, 32, 3, 2, 1), nn.LeakyReLU(inplace=True),
            nn.Conv2d(32, 32, 3, 2, 1), nn.LeakyReLU(inplace=True),
        )
        self.fc1 = nn.Linear(2048, 64)
        self.fc2 = nn.Linear(64, 1)
        nn.init.zeros_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)

    def forward(self, x):
        x_small = F.interpolate(x, size=(256, 256), mode="bilinear", align_corners=False)
        gamma = self.gamma_net(x_small).flatten(1)
        gamma = F.leaky_relu(self.fc1(gamma))
        gamma = self.fc2(gamma)
        gamma = 0.5 + torch.sigmoid(gamma)
        gamma = gamma.view(-1, 1, 1, 1)
        x = torch.clamp(x, 0, 1)
        return x ** gamma


class GatedAGE(nn.Module):
    def __init__(self, brightness_thresh=0.4):
        super().__init__()
        self.brightness_thresh = brightness_thresh
        self.gamma_net = nn.Sequential(
            nn.Conv2d(3, 16, 3, 2, 1), nn.LeakyReLU(inplace=True),
            nn.Conv2d(16, 32, 3, 2, 1), nn.LeakyReLU(inplace=True),
            nn.Conv2d(32, 32, 3, 2, 1), nn.LeakyReLU(inplace=True),
            nn.Conv2d(32, 32, 3, 2, 1), nn.LeakyReLU(inplace=True),
            nn.Conv2d(32, 32, 3, 2, 1), nn.LeakyReLU(inplace=True),
        )
        self.fc1 = nn.Linear(2048, 64)
        self.fc2 = nn.Linear(64, 1)
        nn.init.zeros_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)

    def forward(self, x):
        x_small = F.interpolate(x, size=(256, 256), mode="bilinear", align_corners=False)
        with torch.no_grad():
            brightness = x_small.mean(dim=(1, 2, 3), keepdim=True)
            darkness_gate = torch.clamp(
                1 - brightness / self.brightness_thresh, 0, 1
            )
        gamma = self.gamma_net(x_small).flatten(1)
        gamma = F.leaky_relu(self.fc1(gamma))
        gamma = self.fc2(gamma)
        gamma_raw = 0.5 + torch.sigmoid(gamma)
        gamma_raw = gamma_raw.view(-1, 1, 1, 1)
        gamma = 1.0 + darkness_gate * (gamma_raw - 1.0)
        x = torch.clamp(x, 0, 1)
        return x ** gamma


def register_age(age_class):
    tasks.AGE = age_class
    ns = getattr(tasks, "_cdl_patched_ns", None)
    if ns is not None:
        ns["AGE"] = age_class
    print("Registered AGE as:", age_class.__name__)


## 4. Locate the saved checkpoints

In [ ]:

# These are the run names used by the full experiment notebook.
RUN_NAMES = {
    "R1_baseline": "R1_baseline_scratch",
    "R2_cdl": "R2_cdl_scratch",
    "R3_cdl_aug": "R3_cdl_aug",
    "R4_cdl_age_aug": "R4_cdl_age_aug_v2",
    "R5_cdl_gated_age_aug": "R5_cdl_gated_age_aug",
}

CHECKPOINTS = {
    model: RUNS_DIR / run_name / "weights" / "best.pt"
    for model, run_name in RUN_NAMES.items()
}

for model, ckpt in CHECKPOINTS.items():
    print(f"{model:22s}: {ckpt} ->", "OK" if ckpt.exists() else "MISSING")

missing = [str(p) for p in CHECKPOINTS.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        "One or more saved checkpoints are missing. "
        "This notebook requires the committed best.pt files; it will not retrain them.\n"
        + "\n".join(missing)
    )


## 5. Load checkpoints — no training

In [ ]:

from ultralytics import YOLO

def load_saved_model(model_name):
    # Register the correct AGE implementation before loading R4/R5.
    if model_name == "R4_cdl_age_aug":
        register_age(NaiveAGE)
    elif model_name == "R5_cdl_gated_age_aug":
        register_age(GatedAGE)

    print(f"Loading saved checkpoint for {model_name} ...")
    model = YOLO(str(CHECKPOINTS[model_name]))
    print("Loaded.")
    return model

models = {}
for name in RUN_NAMES:
    models[name] = load_saved_model(name)

print("\nLoaded models:", list(models))



## 6. Reproduce clean / mild-dark / severe-dark evaluation

The evaluation below uses the saved checkpoints and the already-frozen test sets.

**No `model.train()` call appears anywhere in this notebook.**


In [ ]:

import pandas as pd
import numpy as np

EVAL_SETS = {
    "test_clean": CLEAN_YAML,
    "test_dark_mild": MILD_YAML,
    "test_dark_severe": SEVERE_YAML,
}

def evaluate_model(model, model_name, data_yaml):
    res = model.val(
        data=str(data_yaml),
        split="test",
        verbose=False,
    )
    ap = res.box.ap50
    params_m = sum(p.numel() for p in model.model.parameters()) / 1e6
    return {
        "model": model_name,
        "precision": float(res.box.mp),
        "recall": float(res.box.mr),
        "mAP50": float(res.box.map50),
        "mAP50-95": float(res.box.map),
        "AP50_D00": float(ap[0]),
        "AP50_D10": float(ap[1]),
        "AP50_D20": float(ap[2]),
        "AP50_D40": float(ap[3]),
        "params_M": round(params_m, 3),
    }

rows = []
for model_name, model in models.items():
    for test_name, yaml_path in EVAL_SETS.items():
        print(f"\n===== {model_name} | {test_name} =====")
        row = evaluate_model(model, model_name, yaml_path)
        row["test_set"] = test_name
        rows.append(row)
        print(row)

reproduced_df = pd.DataFrame(rows)

model_order = list(RUN_NAMES)
test_order = list(EVAL_SETS)
reproduced_df["model"] = pd.Categorical(
    reproduced_df["model"], categories=model_order, ordered=True
)
reproduced_df["test_set"] = pd.Categorical(
    reproduced_df["test_set"], categories=test_order, ordered=True
)
reproduced_df = reproduced_df.sort_values(["model", "test_set"]).reset_index(drop=True)

reproduced_df


## 7. Main reproduced results

In [ ]:

main_reproduced = reproduced_df.pivot(
    index="model",
    columns="test_set",
    values="mAP50"
).reindex(index=model_order, columns=test_order)

main_reproduced


In [ ]:

# Human-readable summary table.
summary = main_reproduced.copy()
summary.columns = ["Clean mAP50", "Mild-dark mAP50", "Severe-dark mAP50"]
summary



## 8. Validate against the saved reported results

The repository contains `main_results_table.csv` / `results_tables.csv` generated by the original experiment.

The next cell reads the saved results and compares the reported mAP50 values with the values obtained from the committed checkpoints.

Small numerical differences can occur because inference/runtime versions or hardware can affect evaluation details. A large discrepancy should be investigated rather than silently accepted.


In [ ]:

SAVED_MAIN = REPO_ROOT / "main_results_table.csv"
SAVED_RESULTS = REPO_ROOT / "results_tables.csv"

saved_path = SAVED_MAIN if SAVED_MAIN.exists() else SAVED_RESULTS
if not saved_path.exists():
    raise FileNotFoundError("Neither main_results_table.csv nor results_tables.csv was found.")

saved_df = pd.read_csv(saved_path)
print("Comparing with:", saved_path)

# Support either the original model names or the shortened display names.
saved_compare = saved_df.copy()
saved_compare = saved_compare[
    saved_compare["model"].isin(model_order)
][["model", "test_set", "mAP50"]].copy()

comparison = reproduced_df[["model", "test_set", "mAP50"]].merge(
    saved_compare,
    on=["model", "test_set"],
    how="left",
    suffixes=("_reproduced", "_saved"),
)

comparison["absolute_difference"] = (
    comparison["mAP50_reproduced"] - comparison["mAP50_saved"]
).abs()

comparison


In [ ]:

# Highlight any discrepancy larger than a small tolerance.
TOLERANCE = 0.25

if comparison["mAP50_saved"].isna().any():
    print("WARNING: Some saved rows were not found in the repository CSV.")
else:
    bad = comparison[comparison["absolute_difference"] > TOLERANCE]
    if len(bad) == 0:
        print(f"All reproduced mAP50 values are within ±{TOLERANCE} of the saved table.")
    else:
        print(f"WARNING: {len(bad)} value(s) differ by more than ±{TOLERANCE}:")
        display(bad)


## 9. Per-class AP50

In [ ]:

per_class = reproduced_df.pivot_table(
    index=["model", "test_set"],
    values=["AP50_D00", "AP50_D10", "AP50_D20", "AP50_D40"],
    aggfunc="first",
).reindex(
    pd.MultiIndex.from_product(
        [model_order, test_order],
        names=["model", "test_set"]
    )
)

per_class


## 10. Degradation from clean to dark

In [ ]:

pivot = reproduced_df.pivot_table(
    index="model",
    columns="test_set",
    values="mAP50",
    aggfunc="first",
).reindex(index=model_order)

degradation = pd.DataFrame({
    "test_clean": pivot["test_clean"],
    "drop_to_mild": pivot["test_clean"] - pivot["test_dark_mild"],
    "drop_to_severe": pivot["test_clean"] - pivot["test_dark_severe"],
})

degradation



## 11. Optional: verify the saved project result tables

The following cells do **not** rerun the experiment. They simply inspect the CSV artifacts already committed to the repository.


In [ ]:

for filename in [
    "main_results_table.csv",
    "per_class_ap_table.csv",
    "degradation_table.csv",
    "per_country_map_R2_vs_R5.csv",
    "small_defect_subset_table.csv",
]:
    p = REPO_ROOT / filename
    if p.exists():
        print(f"\n### {filename}")
        display(pd.read_csv(p).head(20))
    else:
        print("Missing:", filename)



## 12. Reproduction conclusion

If the checkpoint-loading and evaluation cells run successfully and the reproduced values are consistent with the saved CSV tables, the repository passes the **saved-results reproduction/validation** requirement.

### What this notebook deliberately does NOT do

- It does not download the original RDD2022 release.
- It does not create a new train/validation/test split.
- It does not regenerate the synthetic dark test sets.
- It does not train R1–R5.
- It does not modify the committed checkpoints or experiment artifacts.

For the complete experiment from scratch, use:

`RDD2022_YOLOv11n_CDL+Adaptive_AGE_full_experiment_from_scratch.ipynb`
